# 03 — Real two-GPU throughput: converging queue vs the alternatives

**Run on the CUDA host** (`uv sync --extra dev --extra gpu`, then
`uv run --with jupyterlab --with matplotlib --with datasets jupyter lab`).
This notebook decides whether the project was worth building; keep its
executed outputs committed — they are the evidence. (Committed unexecuted
from the CPU dev machine; execute here and re-commit.)

Four configurations, same inputs, same model:

1. fast GPU alone (TEI-equivalent baseline)
2. both GPUs, static 50/50 item split
3. both GPUs, static split weighted by configured device weights
4. both GPUs, converging queue (embedx as built)

Measurement hygiene, all required: correctness asserted before any timing,
warmup runs discarded, `torch.cuda.synchronize()` on every device around
every timed region, medians over several runs with spread, and an explicit
record of anything else holding VRAM during the run.

In [ ]:
import subprocess
import threading
import time

import numpy as np
import torch

# --- Environment record: contaminated numbers that look clean are worse ---
# --- than no numbers. If Ollama (or anything) holds VRAM, it shows here. ---
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} {torch.cuda.get_device_name(i)}  "
          f"free={free / 2**30:.1f} GiB / total={total / 2**30:.1f} GiB  "
          f"(anything missing from 'free' is held by another process)")
OTHER_VRAM_USERS = "NONE"  # <- EDIT: state what nvidia-smi shows, honestly.
print(f"declared other VRAM users during this run: {OTHER_VRAM_USERS}")

In [ ]:
from embedx.backend.hf import HFBackend, TokenLengthCache
from embedx.config import Settings
from embedx.engine import Engine, make_batches
from embedx.gpu.budgets import device_budgets
from embedx.gpu.discovery import discover_devices, rank_devices

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
settings = Settings(model_id=MODEL, pooling="mean", max_batch_tokens=16384, max_seq_len=512)

infos = discover_devices(settings.devices)
ranked = rank_devices(infos, settings.device_weights)
budgets = device_budgets(ranked, settings.max_batch_tokens, settings.device_batch_tokens)
assert len(ranked) >= 2, "this notebook needs two GPUs"
for d in ranked:
    print(f"[{d.index}] {d.name}  weight={d.weight:.3f}  budget={budgets[d.index]}")

cache = TokenLengthCache()
backends = [
    HFBackend(MODEL, device_index=d.index, pooling=settings.pooling,
              normalize=settings.normalize, dtype=settings.dtype,
              max_seq_length=settings.max_seq_len, length_cache=cache)
    for d in ranked
]
length_fn = backends[0].length_fn
engine = Engine(backends, ranked, settings, length_fn=length_fn)

# Corpus: real texts, real token lengths.
try:
    from datasets import load_dataset
    texts = [r["text"] for r in load_dataset("ag_news", split="train[:8000]")]
    print("corpus: ag_news train[:8000]")
except Exception as exc:  # noqa: BLE001
    rng = np.random.default_rng(7)
    lens = np.clip(rng.lognormal(4.0, 0.9, size=8000).astype(int), 10, 3000)
    texts = ["lorem ipsum " * max(1, n // 12) for n in lens]
    print(f"corpus: SYNTHETIC fallback ({type(exc).__name__})")
token_lengths = [length_fn(t) for t in texts]
TOTAL_TOKENS = sum(token_lengths)
print(f"n={len(texts)}  total tokens={TOTAL_TOKENS}")

In [ ]:
def sync_all():
    for d in ranked:
        torch.cuda.synchronize(d.index)


class Recording:
    # Wraps a backend to record per-device items/tokens/last-finish.
    def __init__(self, inner, device_index):
        self.inner, self.device_index = inner, device_index
        self.dim = inner.dim
        self.reset()

    def reset(self):
        self.items = self.tokens = 0
        self.last_end = 0.0

    def embed(self, batch_texts):
        out = self.inner.embed(batch_texts)
        torch.cuda.synchronize(self.device_index)
        self.items += len(batch_texts)
        self.tokens += sum(length_fn(t) for t in batch_texts)
        self.last_end = time.perf_counter()
        return out


recorders = [Recording(b, d.index) for b, d in zip(backends, ranked)]
rec_engine = Engine(recorders, ranked, settings, length_fn=length_fn)


def run_static_split(shares):
    # shares: fraction of items per device, in rank order. Contiguous split
    # of the length-sorted list; each device runs its own make_batches.
    order = sorted(range(len(texts)), key=lambda i: token_lengths[i])
    bounds = np.cumsum([0] + [int(s * len(texts)) for s in shares])
    bounds[-1] = len(texts)
    threads = []
    for w, (rec, dev) in enumerate(zip(recorders, ranked)):
        chunk = [(i, texts[i]) for i in order[bounds[w]:bounds[w + 1]]]

        def work(rec=rec, dev=dev, chunk=chunk):
            for batch in make_batches(chunk, budgets[dev.index], length_fn=length_fn):
                rec.embed([t for _, t in batch])

        threads.append(threading.Thread(target=work))
    for t in threads:
        t.start()
    for t in threads:
        t.join()


def timed(fn, runs=5, warmup=2):
    # Warmup pays CUDA context + autotune; medians over `runs` with spread.
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        for r in recorders:
            r.reset()
        sync_all()
        t0 = time.perf_counter()
        fn()
        sync_all()
        times.append(time.perf_counter() - t0)
        global_end = time.perf_counter()
        idle = [global_end - r.last_end if r.items else float("nan") for r in recorders]
    med = float(np.median(times))
    return {"median_s": med, "iqr": (float(np.percentile(times, 25)),
                                     float(np.percentile(times, 75))),
            "tokens_per_s": TOTAL_TOKENS / med,
            "per_device": [{"index": d.index, "items": r.items, "tokens": r.tokens,
                            "idle_s": i} for r, d, i in zip(recorders, ranked, idle)]}

In [ ]:
# --- Correctness FIRST. A fast wrong answer is worth nothing. -------------
reference = backends[0].embed(texts[:256])
converged = engine.embed(texts[:256])
np.testing.assert_allclose(converged, reference, atol=1e-3)
print("PASS: converging-engine vectors match single-fast-GPU vectors (atol=1e-3)")

In [ ]:
weights = np.array([d.weight for d in ranked])
configs = {
    "1 fast GPU alone": lambda: [recorders[0].embed([t for _, t in b])
                                 for b in make_batches(list(enumerate(texts)),
                                                       budgets[ranked[0].index],
                                                       length_fn=length_fn)],
    "2 static 50/50": lambda: run_static_split([1 / len(ranked)] * len(ranked)),
    "3 static weighted": lambda: run_static_split(list(weights / weights.sum())),
    "4 converging queue": lambda: rec_engine.embed(texts),
}

results = {}
for name, fn in configs.items():
    results[name] = timed(fn)
    r = results[name]
    print(f"{name:22s} makespan={r['median_s']:.3f}s "
          f"(IQR {r['iqr'][0]:.3f}-{r['iqr'][1]:.3f})  "
          f"{r['tokens_per_s']:,.0f} tok/s")
    for pd in r["per_device"]:
        print(f"    device {pd['index']}: items={pd['items']:5d} "
              f"tokens={pd['tokens']:8d} idle={pd['idle_s']:.3f}s")

In [ ]:
# Idle time is the direct measure of balance: configs 2 and 3 should show a
# clear per-device gap, config 4 should not.
import matplotlib.pyplot as plt

names = list(results)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(names, [results[n]["median_s"] for n in names])
axes[0].set_ylabel("makespan (s), median of 5")
axes[0].tick_params(axis="x", rotation=20)
width = 0.35
for k, d in enumerate(ranked):
    axes[1].bar(np.arange(len(names)) + k * width,
                [results[n]["per_device"][k]["idle_s"] for n in names],
                width, label=f"device {d.index}")
axes[1].set_xticks(np.arange(len(names)) + width / 2, names, rotation=20)
axes[1].set_ylabel("idle time (s)")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Host-to-device transfer per device. --------------------------------
# The A400 sits on PCIe 3.0 x4; torch's static properties cannot see that.
# This measurement is the empirical justification for device_weights.
SIZE_MB = 256
host = torch.empty(SIZE_MB * 2**20, dtype=torch.uint8, pin_memory=True)
for d in ranked:
    torch.cuda.synchronize(d.index)
    times = []
    for _ in range(2):  # warmup
        host.to(f"cuda:{d.index}", non_blocking=True)
        torch.cuda.synchronize(d.index)
    for _ in range(7):
        torch.cuda.synchronize(d.index)
        t0 = time.perf_counter()
        host.to(f"cuda:{d.index}", non_blocking=True)
        torch.cuda.synchronize(d.index)
        times.append(time.perf_counter() - t0)
    gbps = (SIZE_MB / 1024) / np.median(times)
    print(f"device {d.index} ({d.name}): H2D {gbps:.1f} GiB/s "
          f"(median of 7, {SIZE_MB} MiB pinned)")

## Results — fill in from the run above and keep the outputs committed

State the numbers plainly, including if they are small:

- Speedup of **(4) converging queue** over **(1) fast GPU alone**: `__x`
- Speedup of **(4)** over **(3) weighted static split**: `__x`
- Idle-time gap: config 2 `__s` / config 3 `__s` vs config 4 `__s`
- Measured H2D bandwidth ratio between devices: `__`

If the converging queue does not beat the weighted static split by a
meaningful margin on this hardware, that is the result and it goes here
unedited. The honest negative is more useful than a flattering blank.